# GenAI Pipeline — Testing Notebook

LLM-based screening of publications for alternative protein relevance and pillar classification.
Uses Claude with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [1]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from typing import Literal
from pydantic import create_model

load_dotenv()

DB_PATH = "../publications.db"
OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("raw_subsets")

### 2. Data Inspection

In [2]:
EXCEL_PATH = Path("raw_subsets/batch_test_50outML_corr_rand3.xlsx")
df_raw = pd.read_excel(EXCEL_PATH)
df_raw["research_category"] = df_raw["research_category"].str.strip()

df = df_raw[
    (df_raw["scope"] == "in") &
    df_raw["research_category"].notna() &
    (df_raw["research_category"] != "")
].reset_index(drop=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (scope=in, research_category not empty): {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nResearch category distribution:")
print(df["research_category"].value_counts())
df.head()

Raw shape: (2928, 10)
Filtered shape (scope=in, research_category not empty): (1014, 10)

Columns: ['id', 'title', 'abstract', 'year', 'scope', 'pillar', 'research_category', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']

Research category distribution:
research_category
Ingredient optimisation       339
End product formulation       138
Consumer & market research    129
Texturization methods          65
Health & nutrition             60
Other                          52
Food safety & quality          42
Crop development               34
Impact assessments             34
Bioprocess design              33
Feedstocks                     28
Strain development             27
Scaffolding                    14
Cell culture media              8
Cell line development           7
Target molecule selection       4
Name: count, dtype: int64


,id,title,abstract,year,scope,pillar,research_category,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,pub.1192230359,Effect of Plant Protein Ingredients at a Range...,"Hybrid plant and meat (HPM) products, in which...",2025,in,PB,End product formulation,NaN,NaN,End product formulation
1,pub.1183225808,Do ingredients matter? Exploring consumer pref...,It is widely accepted that reducing the consum...,2025,in,PB,Consumer & market research,NaN,NaN,Other
2,pub.1192380853,Low–intensity pulsed electromagnetic field sti...,Pulsed electromagnetic field (PEMF) stimulatio...,2025,in,CM,Texturization methods,NaN,NaN,Crop development
3,pub.1186213778,"Drivers, Barriers, and Innovations in Sustaina...",Sustainable food consumption is crucial for mi...,2025,in,CC,Other,NaN,NaN,Cell culture media
4,pub.1188996314,Temperature-dependent growth kinetics of Lacti...,Abstract Microbial growth responses of probio...,2025,in,PB,Strain development,NaN,NaN,Health & nutrition


In [19]:
df_pb = df[df["pillar"] == "PB"].reset_index(drop=True)
df_f  = df[df["pillar"] == "F"].reset_index(drop=True)
df_cm = df[df["pillar"] == "CM"].reset_index(drop=True)
df_cc = df[df["pillar"] == "CC"].reset_index(drop=True)

all_categories = sorted(df["research_category"].dropna().unique())
breakdown = pd.DataFrame(
    {pillar: grp["research_category"].value_counts().reindex(all_categories, fill_value=0)
     for pillar, grp in [("PB", df_pb), ("F", df_f), ("CM", df_cm), ("CC", df_cc)]},
    index=all_categories,
)
breakdown.index.name = "research_category"
breakdown["Total"] = breakdown.sum(axis=1)
breakdown

,PB,F,CM,CC,Total
research_category,,,,,
Bioprocess design,0,23,9,1,33
Cell culture media,0,0,8,0,8
Cell line development,0,0,7,0,7
Consumer & market research,64,1,32,32,129
Crop development,34,0,0,0,34
End product formulation,126,5,2,5,138
Feedstocks,0,28,0,0,28
Food safety & quality,30,2,3,7,42
Health & nutrition,43,10,0,7,60


### 3. Balanced Subset Creation

Just taking as many as 10 from each research category within each pillar.


In [20]:
RANDOM_STATE=3

def create_balanced_sample(df, category_counts, random_state=RANDOM_STATE):
    """
    category_counts: dict mapping research_category -> n, e.g.
        {"Ingredient optimisation": 5, "End product formulation": 3, "Other": 2}
    Categories with no matching rows are skipped with a warning.
    If n exceeds available rows, all available rows are taken (with a warning).
    """
    samples = []
    for cat, n in category_counts.items():
        subset = df[df["research_category"] == cat]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        if n > available:
            print(f"  Warning: '{cat}' — requested {n} but only {available} available, taking all.")
            n = available
        samples.append(subset.sample(n=n, random_state=random_state))
    combined = pd.concat(samples, ignore_index=True)
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [21]:
test_data_PB = create_balanced_sample(df_pb, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=3)
print(f"test_data_PB: {test_data_PB.shape}")
# test_data_PB[["id", "title", "scope", "pillar"]]

test_data_PB: (100, 10)


In [22]:
test_data_F = create_balanced_sample(df_f, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=3)
print(f"test_data_F: {test_data_F.shape}")
# test_data_F[["id", "title", "scope", "pillar"]]

test_data_F: (79, 10)


In [23]:
test_data_CM = create_balanced_sample(df_cm, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=3)
print(f"test_data_CM: {test_data_CM.shape}")
# test_data_CM[["id", "title", "scope", "pillar"]]

test_data_CM: (67, 10)


In [24]:
test_data_CC = create_balanced_sample(df_cc, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=3)
print(f"test_data_CC: {test_data_CC.shape}")
# test_data_CC[["id", "title", "scope", "pillar"]]

test_data_CC: (58, 10)


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [39]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(initial_test_data, "initial_test_data_rand3.xlsx")
save_subset(test_data_PB, f"rescat_test_data_PB_rand{RANDOM_STATE}.xlsx")
save_subset(test_data_CM, f"rescat_test_data_CM_rand{RANDOM_STATE}.xlsx")
save_subset(test_data_CC, f"rescat_test_data_CC_rand{RANDOM_STATE}.xlsx")
save_subset(test_data_F, f"rescat_test_data_F_rand{RANDOM_STATE}.xlsx")

Saved 100 records to raw_subsets\rescat_test_data_PB_rand3.xlsx
Saved 67 records to raw_subsets\rescat_test_data_CM_rand3.xlsx
Saved 58 records to raw_subsets\rescat_test_data_CC_rand3.xlsx
Saved 79 records to raw_subsets\rescat_test_data_F_rand3.xlsx


### 5. Load Prompt and Select Dataset

In [367]:
AP_PILLAR = "CC"  # ← CHANGE THIS: "PB", "F", "CM", "CC"

DATASETS = {
    "PB": test_data_PB,
    "F":  test_data_F,
    "CM": test_data_CM,
    "CC": test_data_CC,
}
DATASET = DATASETS[AP_PILLAR]
print(f"Pillar: {AP_PILLAR} | Dataset: {DATASET.shape[0]} records")

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC PUBLICATIONS
#ids_to_test = ['pub.1190984789', 'pub.1196086614']
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_rescat_data

Pillar: CC | Dataset: 58 records


In [ ]:
### OPTIONAL: apply corrections to research_category from a spreadsheet of corrections

#CORRECTIONS_PATH = "2_research_category/PB_v5/PB_v5_claude-sonnet-4-6_results_only-v2-haiku-incorrect.xlsx"
#CORRECTIONS_PATH = "2_research_category/F_v5/F_v5_claude-sonnet-4-6_results_D.xlsx" 
#CORRECTIONS_PATH = "2_research_category/CM_v1/CM_v1_claude-sonnet-4-6_results.xlsx" 
#CORRECTIONS_PATH = "2_research_category/CC_v3/CC_v3_claude-sonnet-4-6_results_C.xlsx" 


#corrections = (
#    pd.read_excel(CORRECTIONS_PATH, usecols=["id", "corrected"])
#    .pipe(lambda d: d[d["corrected"].notna() & (d["corrected"].str.strip() != "")])
#    .set_index("id")["corrected"]
#    .str.strip()
#)

DATASET = DATASET.copy()
mask = DATASET["id"].isin(corrections.index)
DATASET.loc[mask, "research_category"] = DATASET.loc[mask, "id"].map(corrections)

print(f"Loaded {len(corrections)} corrections, {mask.sum()} rows updated in DATASET")

Loaded 4 corrections, 4 rows updated in DATASET


In [370]:
PROMPT_VERSION = "v3"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"2_research_category/{AP_PILLAR}_{PROMPT_VERSION}/prompt_rescat_pubs_{AP_PILLAR}_{PROMPT_VERSION}.md"

In [371]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food science research.

Your task is to classify a publication on alternative proteins into a research category based on its title and abstract.

Before assigning categories, identify the novel contribution or primary focus of the paper. The 8 specific categories below (everything except Other) are reserved for publications where that topic is the novel contribution or primary focus of the paper. A category should only be assigned if the paper actively investigates, develops, measures, or reviews something within that domain. For example:
- A paper comparing sensory properties of plant-based, mycoprotein, and cultivated meat products → End product formulation
- A systematic review of extrusion and 3D printing technologies for meat analogues → Texturisation methods
- A paper measuring consumer acceptance of plant-based and cultivated meat products → Consumer & market research
- A paper analysing novel food regulatory frameworks for alternati

### 6. API Call with Prompt Caching

In [372]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [ ]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

PB_CATS = ["Crop development", "Strain development", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
F_CATS  = ["Feedstocks", "Target molecule selection", "Strain development", "Bioprocess design", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
CM_CATS = ["Cell line development", "Cell culture media", "Bioprocess design", "Scaffolding", "End product formulation", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]
CC_CATS = ["Bioprocess design", "Scaffolding", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Consumer & market research", "Impact assessments", "Other"]

PILLAR_CATS = {"PB": PB_CATS, "F": F_CATS, "CM": CM_CATS, "CC": CC_CATS}

def make_schema(cats, include_reasoning):
    # The LLM sometimes returns title-case values (e.g. "Health & Nutrition") while
    # our Literal expects sentence case. cats_map allows case-insensitive lookup so
    # the field_validator can normalise the value before Pydantic validates it.
    cats_map = {c.lower(): c for c in cats}
    cat_type = Literal[*cats]

    class _Base(BaseModel):
        # check_fields=False because "primary"/"secondary" are added by create_model, not defined here
        @field_validator("primary", "secondary", mode="before", check_fields=False)
        @classmethod
        def normalise_case(cls, v):
            if isinstance(v, str):
                return cats_map.get(v.lower(), v)
            return v

    fields = {"primary": (cat_type, ...), "secondary": (cat_type, ...)}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    return create_model("ClassificationSchema", __base__=_Base, **fields)

ClassificationSchema = make_schema(PILLAR_CATS[AP_PILLAR], INCLUDE_REASONING)
print(f"Schema built for {AP_PILLAR}: {list(PILLAR_CATS[AP_PILLAR])}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

Schema built for CC: ['Bioprocess design', 'Scaffolding', 'Ingredient optimisation', 'End product formulation', 'Texturization methods', 'Food safety & quality', 'Health & nutrition', 'Consumer & market research', 'Impact assessments', 'Other']


In [374]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [375]:
def classify_with_error_handling(row, system_prompt):
    pub_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pub_id}: model returned no structured output")
                return {"id": pub_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = pub_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pub_id}: {last_error}")
    return {"id": pub_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [376]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data

In [377]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Run 1 / 1
  [1/58] pub.1190543844
  [2/58] pub.1174613325
  [3/58] pub.1194906364
  [4/58] pub.1184597489
  [5/58] pub.1191849859
  [6/58] pub.1195111619
  [7/58] pub.1194354565
  [8/58] pub.1187276166
  [9/58] pub.1187295059
  [10/58] pub.1196060665
  [11/58] pub.1195609701
  [12/58] pub.1187052760
  [13/58] pub.1193643289
  [14/58] pub.1191284655
  [15/58] pub.1190518754
  [16/58] pub.1187677686
  [17/58] pub.1191637476
  [18/58] pub.1185255386
  [19/58] pub.1194074326
  [20/58] pub.1182396222
  [21/58] pub.1186478980
  [22/58] pub.1194329838
  [23/58] pub.1189349384
  [24/58] pub.1184131962
  [25/58] pub.1184676351
  [26/58] pub.1192359115
  [27/58] pub.1188715770
  [28/58] pub.1189679969
  [29/58] pub.1191584911
  [30/58] pub.1184151966
  [31/58] pub.1187075423
  [32/58] pub.1188616435
  [33/58] pub.1188817720
  [34/58] pub.1191993918
  [35/58] pub.1175608755
  [36/58] pub.1194557570
  [37/58] pub.1186732451
  [38/58] pub.1187944871
  [39/58] pub.1195194673
  [40/58] pub.118427475

,primary_LLM,secondary_LLM,reasoning_LLM,id,status,run
0,Other,Consumer & market research,This paper is primarily a philosophical and co...,pub.1190543844,ok,1
1,End product formulation,Scaffolding,The paper characterises mechanical and textura...,pub.1174613325,ok,1
2,Texturization methods,End product formulation,The review focuses on 3D food printing as a ma...,pub.1194906364,ok,1
3,End product formulation,Health & nutrition,This review systematically compares the nutrit...,pub.1184597489,ok,1
4,Consumer & market research,Other,This paper uses survey data to study consumer ...,pub.1191849859,ok,1
5,Ingredient optimisation,Other,The review focuses primarily on how processing...,pub.1195111619,ok,1
6,Food safety & quality,Other,This review focuses on protein safety assessme...,pub.1194354565,ok,1
7,Other,Health & nutrition,This broad review spans multiple categories (s...,pub.1187276166,ok,1
8,Health & nutrition,Impact assessments,The paper primarily evaluates and compares the...,pub.1187295059,ok,1
9,Health & nutrition,End product formulation,The review's primary focus is on nutrient bioa...,pub.1196060665,ok,1


In [378]:
result_cols = ["id", "run", "primary_LLM", "secondary_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "pillar", "research_category"]].merge(
    results_df[result_cols], on="id", how="left"
)

comparison["primary_correct"] = comparison["research_category"] == comparison["primary_LLM"]
comparison["either_correct"]  = (
    (comparison["research_category"] == comparison["primary_LLM"]) |
    (comparison["research_category"] == comparison["secondary_LLM"])
)

# Overall metrics
n = len(comparison)
print(f"Top-1 accuracy (primary match):  {comparison['primary_correct'].mean():.0%}  (n={n})")
print(f"Top-2 accuracy (either match):   {comparison['either_correct'].mean():.0%}  (n={n})")

# Per-category breakdown
cat_stats = (
    comparison.groupby("research_category")
    .agg(
        n=("primary_correct", "count"),
        primary_correct=("primary_correct", "sum"),
        top2_correct=("either_correct", "sum"),
    )
    .assign(
        primary_acc=lambda d: (d["primary_correct"] / d["n"]).map("{:.0%}".format),
        top2_acc=lambda d: (d["top2_correct"] / d["n"]).map("{:.0%}".format),
    )
)
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "pillar", "research_category",
                "primary_LLM", "secondary_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["primary_correct", "either_correct"]
comparison[display_cols]

Top-1 accuracy (primary match):  91%  (n=58)
Top-2 accuracy (either match):   95%  (n=58)


,n,primary_correct,top2_correct,primary_acc,top2_acc
research_category,,,,,
Bioprocess design,1,1,1,100%,100%
Consumer & market research,10,10,10,100%,100%
End product formulation,4,4,4,100%,100%
Food safety & quality,7,7,7,100%,100%
Health & nutrition,6,6,6,100%,100%
Impact assessments,5,4,5,80%,100%
Ingredient optimisation,9,9,9,100%,100%
Other,13,9,10,69%,77%
Texturization methods,3,3,3,100%,100%


,id,title,abstract,pillar,research_category,primary_LLM,secondary_LLM,reasoning_LLM,primary_correct,either_correct
0,pub.1190543844,Disruptive technologies and intra-value confli...,Synthetic biology is a highly disruptive techn...,CC,Other,Other,Consumer & market research,This paper is primarily a philosophical and co...,True,True
1,pub.1174613325,Mechanical properties and texture profile anal...,"Cultivated meat, or cultured meat, is lab-grow...",CC,End product formulation,End product formulation,Scaffolding,The paper characterises mechanical and textura...,True,True
2,pub.1194906364,Recent Advances in Inks for 3D Food Printing: ...,The integration of 3D printers into food produ...,CC,Texturization methods,Texturization methods,End product formulation,The review focuses on 3D food printing as a ma...,True,True
3,pub.1184597489,Nutritional and Sensory Properties of Meat Ana...,"For centuries, meat has been a staple in the h...",CC,End product formulation,End product formulation,Health & nutrition,This review systematically compares the nutrit...,True,True
4,pub.1191849859,"The Good, the Bad, and the Unsustainable: eval...",Despite the proliferation of alternatives for ...,CC,Consumer & market research,Consumer & market research,Other,This paper uses survey data to study consumer ...,True,True
5,pub.1195111619,Algae Processing: Harnessing a Sustainable Pro...,Plant- and animal-based proteins play a signif...,CC,Ingredient optimisation,Ingredient optimisation,Other,The review focuses primarily on how processing...,True,True
6,pub.1194354565,What makes a food protein unsafe?,With a society increasingly demanding alternat...,CC,Food safety & quality,Food safety & quality,Other,This review focuses on protein safety assessme...,True,True
7,pub.1187276166,Foodomics: A lever to avoid the Darwinian boom...,"Background Since the 18th century, the industr...",CC,Other,Other,Health & nutrition,This broad review spans multiple categories (s...,True,True
8,pub.1187295059,Essential Amino Acids and Fatty Acids in Novel...,Essential amino acids and essential fatty acid...,CC,Health & nutrition,Health & nutrition,Impact assessments,The paper primarily evaluates and compares the...,True,True
9,pub.1196060665,Nutrient Equivalence of Plant-Based and Cultur...,Meat provides high-quality protein and essenti...,CC,Health & nutrition,Health & nutrition,End product formulation,The review's primary focus is on nutrient bioa...,True,True


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign scope and pillar, I need to save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in the 1_prompt_debugging folder.
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results. Includes both metrics and 
6. Write a text document about v1 results and what changes you want to make to the prompt. Repeat from step 1.

In [379]:
save_dir = Path(f"2_research_category/{AP_PILLAR}_{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "top1_accuracy", "value": f"{comparison['primary_correct'].mean():.0%}", "n": n},
    {"metric": "top2_accuracy", "value": f"{comparison['either_correct'].mean():.0%}",  "n": n},
])

out_path = save_dir / f"{AP_PILLAR}_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")

Saved to 2_research_category\CC_v3\CC_v3_claude-sonnet-4-6_results.xlsx


In [324]:
# Records where primary_LLM did not match research_category — for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["primary_correct"], "id"]
incorrect_rescat_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_rescat_data

,id,title,abstract,year,scope,pillar,research_category,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,pub.1187276166,Foodomics: A lever to avoid the Darwinian boom...,"Background Since the 18th century, the industr...",2025,in,CC,Health & nutrition,NaN,NaN,NaN
1,pub.1185255386,Alternative Protein-Based Meat and Fish Analog...,This study aimed to explore the extent of rese...,2025,in,CC,Texturization methods,NaN,NaN,NaN
2,pub.1186478980,Regulatory barriers and incentives for alterna...,Purpose This paper aims to discuss the innovat...,2025,in,CC,Food safety & quality,NaN,NaN,NaN
3,pub.1184131962,Advancements in Research on Alternative Protei...,To ensure food security amid dwindling natural...,2025,in,CC,End product formulation,NaN,NaN,NaN
4,pub.1184151966,"Emerging alternatives to coffee, cocoa and pal...",Despite increasing interest in cellular agricu...,2025,in,CC,Ingredient optimisation,NaN,NaN,NaN
5,pub.1191954992,Exploring the Development of a Clean-Label Veg...,Haematococcus pluvialis and Porphyridium cruen...,2025,in,CC,End product formulation,NaN,NaN,NaN
6,pub.1195166812,Exploring the Potential of Haematococcus pluvi...,The search for sustainable and health-promotin...,2025,in,CC,Ingredient optimisation,NaN,NaN,NaN
7,pub.1196201594,A Review on Recent Processing and Bioprocessin...,A lternative proteins (APs) are anticipated to...,2025,in,CC,Ingredient optimisation,NaN,NaN,NaN
8,pub.1192451356,Bioactive Food Proteins: Bridging Nutritional ...,Bioactive food proteins play multifunctional r...,2025,in,CC,Ingredient optimisation,NaN,NaN,NaN
9,pub.1185588959,How innovation-friendly is the EU novel food r...,The Novel Food Regulation provides the central...,2025,in,CC,Food safety & quality,NaN,NaN,NaN
